# 06b-r — utilizzo causale degli eventi × esposizione ricorsiva × stabilità locale

Questo notebook esegue un solo esperimento informativamente parallelo. Prima confronta un target sinaptico causale con lo stesso target permutato; poi addestra una matrice appaiata 2×2×2: auxiliary off/on, teacher-boundary/pushforward 4 ms e stabilità off/direzionale relativa al passivo. La metrica primaria è il rollout a 8 ms sullo stesso supporto del baseline Hines passivo. Nessun braccio può passare se peggiora il passivo in anche un solo seed.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';DEFAULT_ELM_REF='cbd7be51e87650b7790bf6514309eb065617effa';ELM_REF=os.environ.get('HAYFLOW_ELM_REF',DEFAULT_ELM_REF)
ROOT=Path('/kaggle/working');WORKSPACE=ROOT/'hayflow_workspace';ELM_REPO=WORKSPACE/'elmneuron';WORKSPACE.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();MODULE_PATH=ELM_REPO/'src/hayflow_model/recursive_event_exposure_playground.py';assert MODULE_PATH.is_file(),f'Modulo 06b-r assente nel checkout {REVISION}: {MODULE_PATH}';[sys.modules.pop(name,None) for name in tuple(sys.modules) if name=='src' or name.startswith('src.')];sys.path=[str(ELM_REPO)]+[entry for entry in sys.path if entry!=str(ELM_REPO)];importlib.invalidate_caches();print({'revision':REVISION,'module_exists':MODULE_PATH.is_file()})

## 1. Input immutabili
Servono il dataset composito (base targeted v1.1 + top-up BAP v3) e gli artefatti 05t, 06b-n, 06b-o, 06b-p e 06b-q. La ricerca usa lo SHA-256 dell'indice, quindi funziona anche se Kaggle rinomina lo ZIP `archive.zip`.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source,materialize_nested_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model.effective_membrane_source_playground import EXPECTED_06BN_INDEX_SHA256
from src.hayflow_model.atomic_effective_source_learnability import EXPECTED_06BO_INDEX_SHA256
from src.hayflow_model.event_supported_jump_playground import EXPECTED_06BP_INDEX_SHA256
from src.hayflow_model.recursive_event_exposure_playground import EXPECTED_06BQ_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input');CACHE=Path('/kaggle/working/.06br_nested_inputs')
def indexed(label,expected,env):
 override=os.environ.get(env);source=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override) if override else None)
 if source is None:source=materialize_nested_indexed_artifact_source(INPUT_ROOT,expected,CACHE)
 assert source is not None,f'Artefatto {label} esatto non trovato. Aggiungilo agli Input Kaggle oppure imposta {env}.'
 return source
ARTIFACT_05T_SOURCE=indexed('05t',EXPECTED_05T_INDEX_SHA256,'HAYFLOW_05T_ARTIFACT');ARTIFACT_06BN_SOURCE=indexed('06b-n',EXPECTED_06BN_INDEX_SHA256,'HAYFLOW_06BN_ARTIFACT');ARTIFACT_06BO_SOURCE=indexed('06b-o',EXPECTED_06BO_INDEX_SHA256,'HAYFLOW_06BO_ARTIFACT');ARTIFACT_06BP_SOURCE=indexed('06b-p',EXPECTED_06BP_INDEX_SHA256,'HAYFLOW_06BP_ARTIFACT');ARTIFACT_06BQ_SOURCE=indexed('06b-q',EXPECTED_06BQ_INDEX_SHA256,'HAYFLOW_06BQ_ARTIFACT')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06br_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.'
print({'05t':str(ARTIFACT_05T_SOURCE),'06b-n':str(ARTIFACT_06BN_SOURCE),'06b-o':str(ARTIFACT_06BO_SOURCE),'06b-p':str(ARTIFACT_06BP_SOURCE),'06b-q':str(ARTIFACT_06BQ_SOURCE),'base':str(BASE_SOURCE),'topup':str(TOPUP_SOURCE)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06b-r][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880;print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import RecursiveEventExposureConfig,RecursiveEventExposurePlayground
values=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_recursive_event_exposure_playground.yml').read_text())['recursive_event_exposure_playground'];config=RecursiveEventExposureConfig.from_mapping(values)
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_recursive_event_exposure_playground')
if OUTPUT_DIR.exists():
 assert not (OUTPUT_DIR/'final_report.json').is_file(),f'Risultato completo già presente: {OUTPUT_DIR}. Avvia una sessione nuova per non sovrascriverlo.'
 shutil.rmtree(OUTPUT_DIR);print({'stale_incomplete_output_removed':str(OUTPUT_DIR)})
session=RecursiveEventExposurePlayground(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,ARTIFACT_06BN_SOURCE,ARTIFACT_06BO_SOURCE,ARTIFACT_06BP_SOURCE,ARTIFACT_06BQ_SOURCE,code_revision=REVISION)
contract=session.prepare_recursive_event_exposure_playground();display({'valid':contract['valid'],'factorial_axes':contract['factorial_axes'],'factorial_arm_count':contract['factorial_arm_count'],'parameters':next(iter(contract['parameter_counts'].values())),'auxiliary_contract':contract['causal_auxiliary_contract'],'pushforward_detached':contract['pushforward_previous_prediction_detached']});assert contract['valid'] and not contract['validation_state_accessed'] and not contract['test_state_accessed']

## 2. Probe atomico e matrice biologica 2×2×2
Il probe vero-versus-permutato stabilisce se il target ausiliario è apprendibile. La matrice successiva usa gli stessi seed e gli stessi minibatch per tutti i bracci. I checkpoint sono scelti sulla calibration A; la calibration B sceglie il braccio e applica il veto anti-regressione. Durante l'esecuzione vengono stampati solo progressi compatti, mai tensori o snapshot.

In [ ]:
auxiliary_probe=session.run_causal_auxiliary_probe();display({'valid':auxiliary_probe['valid'],'causal_rmse':auxiliary_probe['causal_target_median_normalized_rmse'],'permuted_rmse':auxiliary_probe['permuted_target_median_normalized_rmse'],'gain':auxiliary_probe['gain_over_permuted_fraction'],'gate_passed':auxiliary_probe['registered_gate_passed']});assert auxiliary_probe['valid']
training=session.train_recursive_factorial_matrix();display({'valid':training['valid'],'diagnostic_best_arm':training['diagnostic_best_arm'],'selected_candidate':training['selected_candidate'],'calibration_8ms_rmse':training['median_calibration_half_B_8ms_rmse_mv'],'eligibility':training['eligibility']});assert training['valid'] and not training['development_used_during_training']

In [ ]:
evaluation=session.evaluate_recursive_factorial_matrix(training);final_report=session.finalize_recursive_event_exposure_playground(contract,auxiliary_probe,training,evaluation)
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'diagnostic_best_arm':final_report['diagnostic_best_arm'],'candidate':final_report['selected_candidate'],'8ms_rmse_mv':final_report['median_development_8ms_rmse_mv'],'support_matched_passive_8ms_rmse_mv':final_report['median_support_matched_passive_8ms_rmse_mv'],'gain_over_passive':final_report['median_gain_over_passive_fraction'],'per_seed_non_regression':final_report['per_seed_non_regression'],'event_materiality':final_report['ordered_event_path_materiality_passed'],'factor_contrasts':final_report['factor_contrasts'],'frozen_06bq_8ms_rmse_mv':final_report['frozen_06bq_reference']['median_8ms_rmse_mv'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['validation_state_accessed'] and not final_report['test_state_accessed']

## 3. Download affidabile
La cella usa il metodo Blob/base64 già validato nel progetto. Non usa `FileLink` né URL `/files/`.

In [ ]:
import base64
from IPython.display import Javascript,display
archive_base=Path('/kaggle/working/hayflow_recursive_event_exposure_playground');archive_path=Path(shutil.make_archive(str(archive_base),'zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(archive_path.read_bytes()).decode('ascii')
javascript=f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{archive_path.name}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),30000);"""
print({'zip':archive_path.name,'size_mib':round(archive_path.stat().st_size/1024**2,2)});display(Javascript(javascript))